# Data Acquisition
**COSC 301 Project — Malaysia State-Level Socioeconomic & Health Outcomes**

This notebook downloads raw data from three sources:
1. **OpenDOSM** : household income, poverty, and population by state
2. **MoH Malaysia GitHub** : public health facilities and hospital beds
3. **World Bank API** : national-level health indicators (backup / cross-validation)

All files are saved to `data/raw/`.

In [1]:
import time
import requests
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

TIMEOUT = 30  # seconds per request


def fetch_json(url: str, params: dict | None = None, retries: int = 3) -> list | dict:
    """GET JSON with timeout and up to `retries` retries on transient failures."""
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=TIMEOUT)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.HTTPError as e:
            raise RuntimeError(f"HTTP error fetching {url}: {e}") from e
        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            if attempt == retries:
                raise RuntimeError(
                    f"Failed to fetch {url} after {retries} attempts: {e}"
                ) from e
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({e}); retrying in {wait}s…")
            time.sleep(wait)


def fetch_csv(url: str, retries: int = 3) -> pd.DataFrame:
    """Download a CSV with timeout and up to `retries` retries."""
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, timeout=TIMEOUT)
            resp.raise_for_status()
            from io import StringIO
            return pd.read_csv(StringIO(resp.text))
        except requests.exceptions.HTTPError as e:
            raise RuntimeError(f"HTTP error fetching {url}: {e}") from e
        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            if attempt == retries:
                raise RuntimeError(
                    f"Failed to fetch {url} after {retries} attempts: {e}"
                ) from e
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({e}); retrying in {wait}s…")
            time.sleep(wait)

## Source 1: OpenDOSM (`api.data.gov.my`)

**License:** Creative Commons Attribution (CC-BY)  
**Docs:** https://open.dosm.gov.my/api-docs

| Dataset | Description |
|---|---|
| `hh_income_parlimen` | Household income by parliament constituency |
| `hh_poverty_parlimen` | Household poverty by parliament constituency |
| `population_state` | State-level population totals (overall age/sex/ethnicity only) for per-capita normalisation |
| `hh_income_state` | Official household income (mean & median) by state, 1970–2022 |
| `hh_poverty_state` | Official household poverty rates by state, 1970–2022 |

In [2]:
# OpenDOSM: Household Income by Parliament Constituency
BASE_DOSM = "https://api.data.gov.my/data-catalogue"
params = {"id": "hh_income_parlimen", "limit": 2000}
df = pd.DataFrame(fetch_json(BASE_DOSM, params=params))
df.to_csv(RAW_DIR / "hh_income_parlimen.csv", index=False)
print(f"✓ hh_income_parlimen: {len(df)} rows")

✓ hh_income_parlimen: 666 rows


In [3]:
# OpenDOSM: Household Poverty by Parliament Constituency
params = {"id": "hh_poverty_parlimen", "limit": 2000}
df = pd.DataFrame(fetch_json(BASE_DOSM, params=params))
df.to_csv(RAW_DIR / "hh_poverty_parlimen.csv", index=False)
print(f"✓ hh_poverty_parlimen: {len(df)} rows")

✓ hh_poverty_parlimen: 666 rows


In [4]:
# OpenDOSM: Population by State (API filter — kept for reference; see cell below for working download)
params = {"id": "population_state", "limit": 10000,
          "filter": "age@overall_age,sex@overall_sex,ethnicity@overall_ethnicity"}
df = pd.DataFrame(fetch_json(BASE_DOSM, params=params))
print(f"  population_state via API filter: {len(df)} rows (known filter issue; direct download below)")

RuntimeError: HTTP error fetching https://api.data.gov.my/data-catalogue: 400 Client Error: Bad Request for url: https://api.data.gov.my/data-catalogue/?id=population_state&limit=10000&filter=age%40overall_age%2Csex%40overall_sex%2Cethnicity%40overall_ethnicity

In [ ]:
# OpenDOSM: Historical Household Income by State (1970–2022)
params = {"id": "hh_income_state", "limit": 2000}
df = pd.DataFrame(fetch_json(BASE_DOSM, params=params))
df.to_csv(RAW_DIR / "hh_income_state.csv", index=False)
print(f"✓ hh_income_state: {len(df)} rows, years {df['date'].str[:4].min()}–{df['date'].str[:4].max()}")

In [ ]:
# OpenDOSM: Historical Household Poverty by State (1970–2022)
params = {"id": "hh_poverty_state", "limit": 2000}
df = pd.DataFrame(fetch_json(BASE_DOSM, params=params))
df.to_csv(RAW_DIR / "hh_poverty_state.csv", index=False)
print(f"✓ hh_poverty_state: {len(df)} rows, years {df['date'].str[:4].min()}–{df['date'].str[:4].max()}")

In [ ]:
# OpenDOSM: Population by State — direct CSV download (API filter returns wrong column names)
URL_DATA = "https://storage.dosm.gov.my/population/population_state.csv"
df = fetch_csv(URL_DATA)
df.to_csv(RAW_DIR / "population_state.csv", index=False)
print(f"✓ population_state: {len(df)} rows")

## Source 2: MoH Malaysia GitHub

**License:** Malaysian Open Data License (free for research)  
**Docs:** https://github.com/MoH-Malaysia

MoH publishes data directly on GitHub as CSVs. Pulling from the official repo is more reliable than scraping the portal.

| File | Description |
|---|---|
| `moh_facilities.csv` | Public hospitals and health clinics by state |
| `moh_beds.csv` | Hospital bed counts and utilisation by facility |

In [ ]:
# MoH Malaysia: Public Facilities
MOH_BASE = "https://raw.githubusercontent.com/MoH-Malaysia/data-resources-public/main"
df = fetch_csv(f"{MOH_BASE}/facilities_master.csv")
df.to_csv(RAW_DIR / "moh_facilities.csv", index=False)
print(f"✓ moh_facilities: {len(df)} rows")

# MoH Malaysia: Hospital Beds
df = fetch_csv(f"{MOH_BASE}/bedutil_facility.csv")
df.to_csv(RAW_DIR / "moh_beds.csv", index=False)
print(f"✓ moh_beds: {len(df)} rows")

## Source 3: World Bank API (National-Level Backup)

**License:** CC-BY 4.0  
**Coverage:** Malaysia (`MYS`), most recent 20 years

National-level indicators for cross-validation and backup.

| Indicator | Column | Description |
|---|---|---|
| `SP.DYN.LE00.IN` | `life_expectancy` | Life expectancy at birth |
| `SP.DYN.IMRT.IN` | `infant_mortality` | Infant mortality (per 1,000 live births) |
| `SH.XPD.CHEX.GD.ZS` | `health_exp_gdp` | Health expenditure (% of GDP) |
| `SH.MED.BEDS.ZS` | `hospital_beds` | Hospital beds (per 1,000 people) |

In [ ]:
WB_BASE = "https://api.worldbank.org/v2/country/MYS/indicator"
WB_PARAMS = {"format": "json", "per_page": 100, "mrv": 20}

indicators = {
    "SP.DYN.LE00.IN": "life_expectancy",
    "SP.DYN.IMRT.IN": "infant_mortality",
    "SH.XPD.CHEX.GD.ZS": "health_exp_gdp",
    "SH.MED.BEDS.ZS": "hospital_beds",
}

frames = []
for code, col in indicators.items():
    data = fetch_json(f"{WB_BASE}/{code}", params=WB_PARAMS)
    rows = [{"year": int(r["date"]), col: r["value"]}
            for r in (data[1] or []) if r["value"] is not None]
    frames.append(pd.DataFrame(rows).set_index("year"))

wb_df = pd.concat(frames, axis=1).reset_index().sort_values("year")
wb_df.to_csv(RAW_DIR / "worldbank_malaysia.csv", index=False)
print(f"✓ worldbank_malaysia: {len(wb_df)} rows")

In [ ]:
# Quick check: all files saved and have data
import glob
files = glob.glob(str(RAW_DIR / "*.csv"))
print(f"\n✓ Downloaded {len(files)} CSV files:")
for f in sorted(files):
    size = len(pd.read_csv(f))
    print(f"  - {Path(f).name}: {size} rows")


✓ Downloaded 8 CSV files:
  - hh_income_parlimen.csv: 666 rows
  - hh_income_state.csv: 303 rows
  - hh_poverty_parlimen.csv: 666 rows
  - hh_poverty_state.csv: 294 rows
  - moh_beds.csv: 149 rows
  - moh_facilities.csv: 3304 rows
  - population_state.csv: 263679 rows
  - worldbank_malaysia.csv: 20 rows
